In [1]:
### set up the notebook
import matplotlib.pyplot as plt
import xarray as xr
import numpy as np
import pandas as pd
import geopandas as gpd
import scipy.signal
from scipy.interpolate import griddata
import dask as da

from IPython.core.display import display, HTML
display(HTML("<style>.container { width:90% !important; }</style>"))
np.set_printoptions(linewidth=100) 

plt.rcParams.update({'font.size': 14})

da.config.set(**{'array.slicing.split_large_chunks': True})

In [10]:
### load the data

ACCESS = xr.open_dataset('../data_to_publish/ACCESS_at_buoys_FSP.nc')
GNSS = xr.open_dataset('../data_to_publish/GNSS_hourly.nc')
ECMWF = xr.open_dataset('../data_to_publish/ECMWF_at_buoys_FSP.nc')

In [11]:
### Reindex GNSS and ACCESS to common time period

new_time = np.arange(np.datetime64('2023-04-01T00:00:00'), np.datetime64('2023-07-01T00:00:00'), np.timedelta64(1, 'h'))

GNSS = GNSS.reindex(time=new_time, method='nearest', tolerance='10 min')
ACCESS = ACCESS.reindex(time=new_time, method='nearest', tolerance='10 min')

In [12]:
### Export data to csv

# create new array with the wet trop for each site in cm
buoys_trop = np.zeros((9, len(new_time)))
buoys_trop[0, :] = GNSS.sn40.values*100
buoys_trop[1, :] = GNSS.sn20.values*100
buoys_trop[2, :] = GNSS.sn06.values*100
buoys_trop[3, :] = GNSS.ss05.values*100
buoys_trop[4, :] = GNSS.ss20.values*100
buoys_trop[5, :] = GNSS.ss30.values*100
buoys_trop[6, :] = GNSS.ss40.values*100
buoys_trop[7, :] = GNSS.swxt.values*100
buoys_trop[8, :] = GNSS.sext.values*100

access_trop = np.zeros((9, len(new_time)))
access_trop[0,:] = ACCESS.sn40.values*100
access_trop[1,:] = ACCESS.sn20.values*100
access_trop[2,:] = ACCESS.sn06.values*100
access_trop[3,:] = ACCESS.ss05.values*100
access_trop[4,:] = ACCESS.ss20.values*100
access_trop[5,:] = ACCESS.ss30.values*100
access_trop[6,:] = ACCESS.ss40.values*100
access_trop[7,:] = ACCESS.swxt.values*100
access_trop[8,:] = ACCESS.sext.values*100

ecmwf_trop = np.zeros((9, len(ECMWF.time)))
ecmwf_trop[0,:] = ECMWF.sn40.values*100*-1
ecmwf_trop[1,:] = ECMWF.sn20.values*100*-1
ecmwf_trop[2,:] = ECMWF.sn06.values*100*-1
ecmwf_trop[3,:] = ECMWF.ss05.values*100*-1
ecmwf_trop[4,:] = ECMWF.ss20.values*100*-1
ecmwf_trop[5,:] = ECMWF.ss30.values*100*-1
ecmwf_trop[6,:] = ECMWF.ss40.values*100*-1
ecmwf_trop[7,:] = ECMWF.swxt.values*100*-1
ecmwf_trop[8,:] = ECMWF.sext.values*100*-1

# where array is 0, set to nan
buoys_trop[buoys_trop == 0] = np.nan
access_trop[access_trop == 0] = np.nan
ecmwf_trop[ecmwf_trop == 0] = np.nan



In [14]:
# save the data as csv files
np.savetxt('../data_to_publish/buoy_hourly_WPD.csv', buoys_trop, delimiter=',')
np.savetxt('../data_to_publish/ACCESS_hourly_WPD.csv', access_trop, delimiter=',')
np.savetxt('../data_to_publish/ECMWF_FSP_WPD.csv', ecmwf_trop, delimiter=',')